In [ ]:
import pandas as pd
import json
import chromadb
from sentence_transformers import SentenceTransformer
from collections import defaultdict
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import os
import re
from scipy.spatial.distance import cosine
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from dotenv import load_dotenv
load_dotenv()

nltk.download('stopwords')
nltk.download('punkt')

In [ ]:
# ---------------------------------------------------------------------------
# Loading 
# ---------------------------------------------------------------------------

def load_chunks_csv(filepath: str) -> list[dict]:
    df = pd.read_csv(filepath)
    chunks = []
    for _, row in df.iterrows():
        chunks.append({
            "metadata": json.loads(row["metadata"]),
            "text":     row["text"]
        })
    print(f"Loaded {len(chunks)} chunks from {filepath}")
    return chunks


def load_all_categories(category_paths: dict[str, str]) -> dict[str, list[dict]]:
    """
    category_paths: {"Materials": "processed_chunks/material_chunks.csv", ...}
    Returns: {"Materials": [chunk, chunk, ...], ...}
    """
    category_chunks = {}
    for category_name, path in category_paths.items():
        category_chunks[category_name] = load_chunks_csv(path)
    return category_chunks

In [ ]:
# ---------------------------------------------------------------------------
# Stage 1: Build category profiles
# ---------------------------------------------------------------------------

EPD_TOKENS = json.loads(os.getenv("EPD_TOKENS"))
MATERIALS_TOKENS = json.loads(os.getenv("EPD_TOKENS"))
CERTIFICATION_TOKENS = json.loads(os.getenv("CERTIFICATION_TOKENS"))
STANDARDS_TOKENS = json.loads(os.getenv("STANDARDS_TOKENS"))
OTHER_TAGS_TOKENS = json.loads(os.getenv("OTHER_TAGS_TOKENS"))
SCORES_TOKENS = json.loads(os.getenv("SCORES_TOKENS"))
ATTRIBUTE_DESCRIPTIONS = json.loads(os.getenv("ATTRIBUTE_DESCRIPTIONS"))
ATTRIBUTE_TOKENS = [
    token
    for name, description in ATTRIBUTE_DESCRIPTIONS.items()
    for token in (name, description)
]
PRODUCT_PERFORMANCE_TOKENS = []
PRODUCT_CORE_TOKENS = json.loads(os.getenv("PRODUCT_CORE_TOKENS"))
PRODUCT_COMPLIANCE_TOKENS = json.loads(os.getenv("PRODUCT_COMPLIANCE_TOKENS"))
PRODUCT_PROPERTIES_TOKENS = json.loads(os.getenv("PRODUCT_PROPERTIES_TOKENS"))

In [ ]:
CATEGORY_TOKENS = json.loads(os.getenv("CATEGORY_TOKENS"))

model  = SentenceTransformer("all-MiniLM-L6-v2")
CATEGORY_TOKEN_EMBEDDINGS = {
    category_name: [
        model.encode(token, normalize_embeddings=True)
        for token in tokens
    ]
    for category_name, tokens in CATEGORY_TOKENS.items()
}

for category_name, embeddings in CATEGORY_TOKEN_EMBEDDINGS.items():
    np.save(f"category_profiles/{category_name}_embeddings.npy", np.array(embeddings))

# Save token lists too so you can match indices back to token strings later
import json
with open("category_profiles/tokens.json", "w") as f:
    json.dump(CATEGORY_TOKENS, f)

In [ ]:
# ---------------------------------------------------------------------------
# Stage 2: Compute query-aware weights
# ---------------------------------------------------------------------------

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def compute_category_weight(
    query: str,
    query_embedding: np.ndarray,
    tokens: list[str],
    token_embeddings: np.ndarray | None,
    chunk_embeddings: np.ndarray | None,
    strategy: str,
    exact_weight: float = 0.5,
    semantic_weight: float = 0.5,
    top_k: int = 3,
    semantic_threshold: float = 0.1,
) -> float:

    query_lower = query.lower()

    # --- Exact match (always against tokens if available) ---
    exact_score = 0.0
    if tokens:
        matched = []
        for token in tokens:
            token_lower = token.lower()
            if token_lower in query_lower:
                matched.append(token)
            else:
                first_word = token_lower.split()[0]
                if len(first_word) >= 3 and first_word in query_lower:
                    matched.append(token)
        if matched:
            exact_score = min(1.0, 0.7 + (len(matched) - 1) * 0.1)

    # --- Semantic: pick source based on strategy ---
    if strategy == "tokens" and token_embeddings is not None:
        semantic_embeddings = token_embeddings
    elif strategy == "chunks" and chunk_embeddings is not None:
        semantic_embeddings = chunk_embeddings   # chunks for semantic, tokens for exact
    else:
        semantic_embeddings = token_embeddings   # fallback

    sims = [cosine_sim(query_embedding, emb) for emb in semantic_embeddings]
    sims_thresholded = [s if s >= semantic_threshold else 0.0 for s in sims]
    semantic_score = float(np.mean(sorted(sims_thresholded, reverse=True)[:top_k]))

    return (exact_weight * exact_score) + (semantic_weight * semantic_score)

def compute_all_category_weights(
    query: str,
    category_profile_strategy_dict: dict,
    loaded_token_embeddings_dict: dict,
    loaded_chunk_embeddings_dict: dict
) -> dict[str, float]:
    query_embedding = model.encode(query, normalize_embeddings=True)
    weights = {}

    for category_name, strategy in category_profile_strategy_dict.items():
        tokens = CATEGORY_TOKENS.get(category_name, [])
        token_embeddings = loaded_token_embeddings_dict.get(category_name, None)
        chunk_embeddings = loaded_chunk_embeddings_dict.get(category_name, None)

        weights[category_name] = compute_category_weight(
            query=query,
            query_embedding=query_embedding,
            tokens=tokens,
            token_embeddings=token_embeddings,
            chunk_embeddings=chunk_embeddings,
            strategy=strategy,
        )

    return weights

In [ ]:
# Which categories use which strategy
CATEGORY_PROFILE_STRATEGY = {
    "epd":                  "tokens",
    "materials":            "tokens",
    "certifications":       "tokens",
    "attributes":           "tokens",
    "standards":            "tokens",
    "tags":                 "tokens",
    "product_core":         "tokens",
    "product_performance":  "chunks",
    "product_properties":   "tokens",
    "product_compliance":   "tokens",
    "scores":               "tokens",
}

# ----------------------------------------------------------------
# CHROMADB — chunk embeddings
# ----------------------------------------------------------------

client = chromadb.PersistentClient(path="./chroma_db")

collections = {
    name: client.get_or_create_collection(name=name)
    for name in CATEGORY_PROFILE_STRATEGY.keys()
}

def load_chunk_embeddings_from_chroma(collections: dict) -> dict[str, np.ndarray]:
    chunk_embeddings = {}
    for category_name, collection in collections.items():
        results = collection.get(include=["embeddings"])
        embeddings = results["embeddings"]
        if embeddings is not None:
            chunk_embeddings[category_name] = np.array(embeddings)
            print(f"Loaded {len(embeddings)} chunk embeddings for {category_name}")
        else:
            print(f"Warning: no embeddings found for {category_name}")
    return chunk_embeddings

LOADED_CHUNK_EMBEDDINGS = load_chunk_embeddings_from_chroma(collections)

# ----------------------------------------------------------------
# NPY FILES — token embeddings
# ----------------------------------------------------------------

def load_token_embeddings(collections: dict) -> dict[str, np.ndarray]:
    token_embeddings = {}
    for category_name, _ in collections.items():
        path = f"category_profiles/{category_name}_embeddings.npy"
        if os.path.exists(path):
            token_embeddings[category_name] = np.load(path)
            print(f"Loaded token embeddings for {category_name}")
        else:
            print(f"Warning: no token embeddings found for {category_name}")
    return token_embeddings

LOADED_TOKEN_EMBEDDINGS = load_token_embeddings(CATEGORY_PROFILE_STRATEGY)

In [ ]:
# ---------------------------------------------------------------------------
# Stage 3: Apply weights to hybrid scores
# ---------------------------------------------------------------------------
def compute_final_score(hybrid_scores_per_category: dict[str, float],
                          category_weights: dict[str, float],
                          weight_threshold=0.05) -> float:
    numerator = 0.0
    denominator = 0.0
    for category_name, score in hybrid_scores_per_category.items():
        w_c = category_weights.get(category_name, 0.0)
        if w_c < weight_threshold:   # ignore categories the query doesn't care about
            continue
        numerator += w_c * score
        denominator += w_c

    if denominator == 0:
        return 0.0

    return numerator / denominator

def normalize_weights(weights: dict[str, float]) -> dict[str, float]:
    values = np.array(list(weights.values()))
    exp_values = np.exp(values - np.max(values))  # subtract max for numerical stability
    softmax_values = exp_values / exp_values.sum()
    return dict(zip(weights.keys(), softmax_values.tolist()))

In [ ]:
# ------------------------------------------------------------------ #
# 1. SINGLE ATTRIBUTE — tests if individual collections retrieve well #
# ------------------------------------------------------------------ #
SINGLE_ATTRIBUTE_TEST_QUERIES = [
    # material
    "nylon polyamide carpet flooring",
    "cross laminated timber CLT",
    # tags
    "slip resistant R11 flooring",
    "fire safe material",
    # attributes
    "carbon negative product",
    "low toxicity non hazardous product",
    # certifications
    "EPD certified product",
    "C2C Bronze certified material",
    # standards
    "ISO 14001 certified product",
    "BES 6001 responsible sourcing standard",
    # compliance
    "BREEAM Mat01 life cycle impacts",
    "LEED optimize energy performance",
    # epd
    "low global warming potential during production",
    "low non-renewable energy during production",
    # scores
    "exceptional sustainability performance product",
    "product with positive health benefit",
]

# ------------------------------------------------------------------ #
# 2. MULTI ATTRIBUTE — tests cross-collection merging                 #
# ------------------------------------------------------------------ #
MULTI_ATTRIBUTE_TEST_QUERIES = [
    # strong
    "glass wool acoustic panel 40mm residential Sweden",
    "polyurethane foam board insulation EPD certified France exceptional sustainability",
    "porcelain fire resistant Euroclass A1 Spain sustainability",
    "steel carbon steel UKCA CE certified UK exceptional sustainability BES 6001",
    "glass wool EPD Eurofins indoor air comfort BREEAM Mat01",
    # weak
    "sustainable construction material France",
    "glass product recyclable daylighting",
    "fire resistant certified building material",
    "indoor air quality certified product residential",
    "plant-based sustainable low VOC",
    #no
    "cross laminated timber structural panel",
    "concrete ready mix C30 admixture",
    "cellulose blown-in insulation recycled paper",
    "waterproofing membrane EPDM flat roof",
    "LED luminaire emergency lighting DALI",
]

# ------------------------------------------------------------------ #
# 7. NATURAL LANGUAGE / INTENT — tests semantic understanding        #
# these are how real users actually search                           #
# ------------------------------------------------------------------ #
NATURAL_LANGUAGE_TEST_QUERIES = {

    # ------------------------------------------------------------------ #
    # CASUAL USERS — broad intent, minimal technical knowledge            #
    # ------------------------------------------------------------------ #
    "casual": [
        "I need a carpet that is easy to clean and durable",
        "what vinyl flooring is safe for a bathroom or wet room",
        "looking for an eco friendly linoleum floor made from natural materials",
        "what bricks are weather resistant and long lasting for an exterior wall",
        "what plasterboard is fire resistant and moisture proof",
        "what blocks are suitable for an energy efficient wall",
        "what acoustic panel reduces noise in an open plan office",
        "looking for a ceiling panel that absorbs sound and is recyclable",
        "I need a hanging acoustic panel for a meeting room",
        "what insulation keeps heat in and is safe to install",
        "looking for a thermal insulation that is also fire safe",
    ],

    # ------------------------------------------------------------------ #
    # TECHNICAL USERS — architects, consultants, contractors, procurement #
    # ------------------------------------------------------------------ #
    "technical": [
        "I need a flooring with an EPD that helps with our BREEAM credits",
        "what acoustic panels contribute to BREEAM indoor environment quality",
        "looking for FSC certified timber that qualifies for LEED wood sourcing credits",
        "we need a modular product designed for disassembly for our BREEAM Mat06 submission",
        "what facade systems have low embodied carbon with a verified EPD",
        "what insulation has low thermal conductivity and low embodied energy",
        "show me products with verified low carbon and strong environmental performance score",
        "what waterproof membranes work for swimming pools and wet rooms",
        "looking for products from Austrian manufacturers with strong sustainability scores",
        "what products have a valid EPD and comply with EN 15804",
    ],
}

In [ ]:
def compute_semantic_scores_from_chroma(query: str, collection) -> dict[str, float]:
    """
    Query a ChromaDB collection semantically.
    Returns {product_id: semantic_score}
    """
    query_embedding = model.encode(query, normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=collection.count(),
        include=["metadatas", "distances", "documents"]
    )

    scores = {}
    for i, (metadata, document) in enumerate(zip(results["metadatas"][0], results["documents"][0])):
        product_id = str(metadata["product_id"])
        similarity = 1 - results["distances"][0][i]
        # if similarity < 0.1:
        #     continue
        if product_id not in scores or similarity > scores[product_id]["semantic"]:
            if (product_id == "8048"):
                print(f"PID {product_id}: DOCUMENT {document}")
            scores[product_id] = {
                "semantic": similarity,
                "text":     document,
            }

    return scores

def compute_token_match_score(query: str, product_text: str) -> tuple[list, float]:
    def tokenize(text):
        return set(re.sub(r'[^\w\s]', '', text.lower()).split())
    
    stop_words = set(stopwords.words('english'))
    query_tokens = tokenize(query)
    text_tokens  = tokenize(product_text)

    filtered_query_tokens = set(word for word in query_tokens if word not in stop_words)
    filtered_text_tokens  = set(word for word in text_tokens  if word not in stop_words)
    matched      = filtered_query_tokens & filtered_text_tokens

    # filter out short noise tokens (articles, prepositions, etc.)
    matched = {t for t in matched if len(t) > 2}

    if not matched:
        return [], 0.0

    score = min(1.0, round(0.7 + (len(matched) - 1) * 0.1, 4))
    return sorted(matched), score

def compute_token_scores_from_collection(query: str, collection) -> dict[str, tuple[list, float]]:
    """
    Scan every document in the collection and compute token match score.
    Returns {product_id: (matched_tokens, score)}
    """
    all_docs = collection.get(include=["documents", "metadatas"])

    token_scores = {}
    for document, metadata in zip(all_docs["documents"], all_docs["metadatas"]):
        product_id = str(metadata["product_id"])
        matched, score = compute_token_match_score(query, document)

        # keep best score if product has multiple chunks
        if product_id not in token_scores or score > token_scores[product_id][1]:
            token_scores[product_id] = (matched, score)

    return token_scores

def compute_category_scores_for_product(
    query: str,
    collection,
    alpha: float = 0.5
) -> dict[str, dict]:

    semantic_scores = compute_semantic_scores_from_chroma(query, collection)
    token_scores = compute_token_scores_from_collection(query, collection)
    
    # union of pids from both signals
    all_pids = set(semantic_scores.keys())
    all_pids |= set(token_scores.keys())

    combined = {}
    for product_id in all_pids:
        sem   = semantic_scores.get(product_id, {})
        s_sem = sem.get("semantic", 0.0)
        s_text = sem.get("text", "")

        matched, s_tok = token_scores.get(product_id, ([], 0.0))
        hybrid = alpha * s_sem + (1 - alpha) * s_tok

        if hybrid < 0.1:
            continue

        combined[product_id] = {
            "hybrid":        hybrid,
            "semantic":      s_sem,
            "semantic_text": s_text,
            "token":         s_tok,
            "token_text":    ", ".join(sorted(matched)),
        }            

    return combined


def compute_all_hybrid_scores(query: str, alpha: float = 0.5) -> dict[str, dict[str, float]]:
    """
    Compute hybrid scores across all categories.
    Returns {product_id: {category_name: score}}
    """
    all_scores = {}
    for category_name, collection in collections.items():
        category_scores = compute_category_scores_for_product(
            query, collection, alpha=alpha
        )
        for product_id, score_dict in category_scores.items():
            if product_id not in all_scores:
                all_scores[product_id] = {}
            if score_dict["hybrid"] >= 0.1:
                all_scores[product_id][category_name] = score_dict

    return all_scores

def search_hybrid(query: str, top_k: int = 5, alpha: float = 0.5) -> list:

    # Phase 1 — compute hybrid scores per product per category
    all_scores = compute_all_hybrid_scores(query, alpha=alpha)

    if not all_scores:
        return []

    # Phase 2 — fetch documents for snippets (same as original Phase 2)
    product_scores = {}
    for pid, category_scores in all_scores.items():
        weighted_sum = sum(s["hybrid"] for cat, s in category_scores.items())
        total_weight = sum(1.0 for cat in category_scores)
        product_scores[pid] = (weighted_sum / total_weight) if total_weight > 0 else 0.0

    ranked = sorted(product_scores.items(), key=lambda x: x[1], reverse=True)

    # Collect top_k unique scores, including all items tied at the cutoff rank
    results = []
    unique_scores_seen = []

    for pid, score in ranked:
        rounded_score = round(score, 4)

        if rounded_score not in unique_scores_seen:
            if len(unique_scores_seen) >= top_k:
                break  # Already have top_k distinct score levels
            unique_scores_seen.append(rounded_score)

        results.append({
            "product_id":      pid,
            "score":           rounded_score,
            "category_scores": all_scores[pid],
        })

    return results

combined_df = pd.read_csv("processed_chunks/combined_chunks.csv")
combined_df["product_id"] = combined_df["product_id"].astype(str)
product_lookup = combined_df.set_index("product_id").to_dict(orient="index")

def results_to_dataframe(results: list, query: str, df_rows: list):
    if not results:
        df_rows.append({"query": query})
        return

    for r in results:
        row = {
            "query":      query,
            "product_id": str(r["product_id"]),
            "score":      r["score"],
        }

        for cat, s in r["category_scores"].items():
            row[f"text_{cat}"]     = s.get("semantic_text", "")
            row[f"sim_{cat}"]      = round(s["semantic"], 4)
            row[f"hybrid_{cat}"]   = round(s["hybrid"], 4)
            row[f"token_{cat}"]    = round(s.get("token", 0.0), 4)
            row[f"token_text_{cat}"] = s.get("token_text", "")

        # matches: one entry per category, sorted by hybrid score
        match_summaries = []
        for cat, s in sorted(r["category_scores"].items(), key=lambda x: x[1]["hybrid"], reverse=True):
            match_summaries.append(
                f"[{cat}]"
                f" | H:{round(s['hybrid'], 4)}"
                f" S:{round(s['semantic'], 4)}"
                f" T:{round(s.get('token', 0.0), 4)}"
                f" | matched_tokens: {s.get('token_text', '')}"
                f" | {s.get('semantic_text', '')[:150]}"
            )
        row["matches"] = " ;; ".join(match_summaries)

        df_rows.append(row)

In [ ]:
def run_test_queries(test_queries: list, top_k: int = 5, csv_file_name: str = "search_results"):
    results_log = {}
    all_rows = []

    for query in test_queries:
        results = search_hybrid(query, top_k=top_k)
        results_log[query] = results
        # print_results(results, query)
        results_to_dataframe(results, query, all_rows)

    df = pd.DataFrame(all_rows)
    df.to_csv(f"{csv_file_name}.csv", index=False)

    return results_log

# model  = SentenceTransformer("all-MiniLM-L6-v2")
# results_log = run_test_queries(SINGLE_ATTRIBUTE_TEST_QUERIES, csv_file_name="results/hybrid_single_attr_results")
# results_log = run_test_queries(MULTI_ATTRIBUTE_TEST_QUERIES, csv_file_name="results/hybrid_multi_attr_results")
results_log = run_test_queries(NATURAL_LANGUAGE_TEST_QUERIES["casual"], csv_file_name="results/hybrid_NL_casual_results")
# results_log = run_test_queries(NATURAL_LANGUAGE_TEST_QUERIES["technical"], csv_file_name="results/hybrid_NL_technical_results")

### Compute Query-Aware Scores

In [ ]:
def search(query: str, top_n: int = 10, score_threshold: float = 0.1) -> list[tuple[str, float]]:

    # Stage 2 — category weights, once per query
    raw_weights = compute_all_category_weights(
        query,
        CATEGORY_PROFILE_STRATEGY,
        LOADED_TOKEN_EMBEDDINGS,
        LOADED_CHUNK_EMBEDDINGS
    )
    normalized_weights = normalize_weights(raw_weights)

    # Stage 3a — hybrid scores per product per category, once per query
    all_product_hybrid_scores = compute_all_hybrid_scores(query)

    # Stage 3b — apply weights to get final score per product
    results = []
    for product_id, hybrid_scores in all_product_hybrid_scores.items():
        score = compute_final_score(hybrid_scores, normalized_weights)
        if score >= score_threshold:
            results.append((product_id, score))

    results.sort(key=lambda x: -x[1])
    return results[:top_n]

# ---------------------------------------------------------------------------
# End-to-end usage
# ---------------------------------------------------------------------------
top_products = search("fire resistant FSC certified acoustic insulation", top_n=10)
for rank, (product_id, score) in enumerate(top_products, start=1):
    print(f"{rank}. Product {product_id} — score: {score:.4f}")